In [1]:
import random
import torch
import os
import numpy as np
import pandas as pd
import polars as pl

In [2]:
INPUT_DIR = '.'

In [3]:
# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    pl.set_random_seed(random_state)
    pd.core.common.random_state(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state

RANDOM_STATE = set_random_states(1618)

In [4]:
def make_aggregated_outputs(input_file, metrics = ['accuracy', 'precision', 'recall', 'f1', 'kappa', 'MCC']):
    dimensions = [
        'informational_vs_involved',
        'non-narrative_vs_narrative',
        'situation-dependent_vs_explicit',
        'non-persuasive_vs_persuasive',
        'non-abstract_vs_abstract',
        'compressed_vs_elaborated'
    ]

    saved_output_dict = {}

    for folder in os.listdir(INPUT_DIR):
        folder_path = os.path.join(INPUT_DIR, folder)

        if not (os.path.isdir(folder_path) and 'outputs' in folder):
            continue

        saved_output_dict[folder] = {
            dim: {metric: [] for metric in metrics}
            for dim in dimensions
        }

        for sub_folder in os.listdir(folder_path):
            sub_path = os.path.join(folder_path, sub_folder)

            if not (os.path.isdir(sub_path) and sub_folder.isdigit()):
                continue

            file_path = os.path.join(sub_path, f"{input_file}.csv")
            df = pd.read_csv(file_path)

            for dimension in dimensions:
                temp_df = df[df['dimension'] == dimension]

                if temp_df.empty:
                    raise ValueError(f"There should be something in the temp_df for the dimension {dimension}.")

                row = temp_df.iloc[0]

                for metric in metrics:
                    saved_output_dict[folder][dimension][metric].append(float(row[metric]))

    temp_dict_all = {}

    for folder, dimensions_dict in saved_output_dict.items():
        series_list = []

        for dimension, metrics_dict in dimensions_dict.items():
            mean_series = pd.Series({
                metric: (sum(values) / len(values)) if values else float('nan')
                for metric, values in metrics_dict.items()
            }, name=dimension)

            series_list.append(mean_series)

        df_folder = pd.concat(series_list, axis=1)
        temp_dict_all[folder] = df_folder

    df_all_folders = pd.concat(temp_dict_all, axis=0)

    df_mean_all = df_all_folders.groupby(level=1).mean()

    return df_all_folders, df_mean_all

In [5]:
all_classif, mean_classif = make_aggregated_outputs('classification_comparison_results_zero_vs_biber')

In [6]:
all_classif

informational_vs_involved  non-narrative_vs_narrative  \
outputsTrain accuracy                    0.648700                    0.512400   
             precision                   0.560383                    0.415947   
             recall                      0.705705                    0.524434   
             f1                          0.623589                    0.462901   
             kappa                       0.302686                    0.027507   
             MCC                         0.310831                    0.028083   
outputsTest  accuracy                    0.641400                    0.503700   
             precision                   0.552629                    0.403622   
             recall                      0.688017                    0.513427   
             f1                          0.612095                    0.450912   
             kappa                       0.286277                    0.010073   
             MCC                         0.292818                    0.010090   
outputsAll   accuracy                    0.642778                    0.502037   
             precision                   0.553941                    0.407520   
             recall                      0.695765                    0.514502   
             f1                          0.616066                    0.453444   
             kappa                       0.290060                    0.008137   
             MCC                         0.297273                    0.008199   

                        situation-dependent_vs_explicit  \
outputsTrain accuracy                          0.597200   
             precision                         0.666464   
             recall                            0.609374   
             f1                                0.635749   
             kappa                             0.186842   
             MCC                               0.188391   
outputsTest  accuracy                          0.599000   
             precision                         0.669164   
             recall                            0.607651   
             f1                                0.636062   
             kappa                             0.191628   
             MCC                               0.193236   
outputsAll   accuracy                          0.595741   
             precision                         0.667273   
             recall                            0.598569   
             f1                                0.630320   
             kappa                             0.186353   
             MCC                               0.188310   

                        non-persuasive_vs_persuasive  \
outputsTrain accuracy                       0.562000   
             precision                      0.414902   
             recall                         0.600057   
             f1                             0.489296   
             kappa                          0.128112   
             MCC                            0.135520   
outputsTest  accuracy                       0.560500   
             precision                      0.418187   
             recall                         0.599373   
             f1                             0.491342   
             kappa                          0.125639   
             MCC                            0.132555   
outputsAll   accuracy                       0.564074   
             precision                      0.415250   
             recall                         0.596155   
             f1                             0.488076   
             kappa                          0.130083   
             MCC                            0.137370   

                        non-abstract_vs_abstract  compressed_vs_elaborated  
outputsTrain accuracy                   0.539900                  0.454600  
             precision                  0.259861                  0.130284  
             recall                     0.609435                  

In [7]:
mean_classif

,informational_vs_involved,non-narrative_vs_narrative,situation-dependent_vs_explicit,non-persuasive_vs_persuasive,non-abstract_vs_abstract,compressed_vs_elaborated
MCC,0.300307,0.015457,0.189979,0.135148,0.100937,-0.133849
accuracy,0.644293,0.506046,0.597314,0.562191,0.537364,0.454880
f1,0.617250,0.455752,0.634044,0.489571,0.358405,0.191510
kappa,0.293008,0.015239,0.188274,0.127945,0.082858,-0.105269
precision,0.555651,0.409030,0.667633,0.416113,0.256791,0.132927
recall,0.696496,0.517454,0.605198,0.598528,0.602927,0.348577


In [8]:
all_contin, mean_contin = make_aggregated_outputs('continuous_comparison_results_zero_vs_biber', ['pearson', 'spearman', 'MSE', 'RMSE', 'MAE'])

In [9]:
all_contin

informational_vs_involved  non-narrative_vs_narrative  \
outputsTrain pearson                    0.360418                    0.045193   
             spearman                   0.363282                    0.065554   
             MSE                        1.279164                    1.909614   
             RMSE                       1.128810                    1.380355   
             MAE                        0.905330                    1.064162   
outputsTest  pearson                    0.340047                    0.022886   
             spearman                   0.341926                    0.046089   
             MSE                        1.319905                    1.954229   
             RMSE                       1.147164                    1.396267   
             MAE                        0.910399                    1.075047   
outputsAll   pearson                    0.347333                    0.025445   
             spearman                   0.346515                    0.053591   
             MSE                        1.305333                    1.949111   
             RMSE                       1.141105                    1.394741   
             MAE                        0.912482                    1.074890   

                       situation-dependent_vs_explicit  \
outputsTrain pearson                          0.163029   
             spearman                         0.246907   
             MSE                              1.673941   
             RMSE                             1.291853   
             MAE                              0.985738   
outputsTest  pearson                          0.148075   
             spearman                         0.243782   
             MSE                              1.703849   
             RMSE                             1.303225   
             MAE                              0.992069   
outputsAll   pearson                          0.162199   
             spearman                         0.249752   
             MSE                              1.675603   
             RMSE                             1.292664   
             MAE                              0.979977   

                       non-persuasive_vs_persuasive  non-abstract_vs_abstract  \
outputsTrain pearson                       0.136737                  0.093741   
             spearman                      0.162127                  0.157311   
             MSE                           1.726525                  1.812519   
             RMSE                          1.312104                  1.343738   
             MAE                           0.985838                  1.003216   
outputsTest  pearson                       0.139873                  0.090411   
             spearman                      0.154970                  0.150319   
             MSE                           1.720255                  1.819177   
             RMSE                          1.309531                  1.346477   
             MAE                           0.989904                  1.006713   
outputsAll   pearson                       0.155894                  0.079038   
             spearman                      0.165949                  0.136905   
             MSE                           1.688211                  1.841924   
             RMSE                          1.297949                  1.354422   
             MAE                           0.981161                  1.004795   

                       compressed_vs_elaborated  
outputsTrain pearson                  -0.063600  
             spearman                 -0.150099  
             MSE                       2.127201  
             RMSE                      1.456838  
             MAE                       1.097015  
outputsTest  pearson                  -0.053718  
             spearman                 -0.151415  
             MSE                       2.107435  
             RMSE                      1.450407  
             MAE

In [10]:
mean_contin

,informational_vs_involved,non-narrative_vs_narrative,situation-dependent_vs_explicit,non-persuasive_vs_persuasive,non-abstract_vs_abstract,compressed_vs_elaborated
MAE,0.909404,1.071366,0.985928,0.985635,1.004908,1.096089
MSE,1.301467,1.937651,1.684464,1.711664,1.824540,2.107706
RMSE,1.139026,1.390454,1.295914,1.306528,1.348212,1.450284
pearson,0.349266,0.031174,0.157768,0.144168,0.087730,-0.053853
spearman,0.350574,0.055078,0.246814,0.161015,0.148178,-0.142738
